In [1]:
from ngsolve import *
from netgen.geom2d import unit_square
import json
import numpy as np
import os
import matplotlib
import time
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.family": "serif",
    # Put a guaranteed Matplotlib serif first to avoid sans fallback.
    "font.serif": ["DejaVu Serif", "CMU Serif", "Computer Modern Roman", "Latin Modern Roman"],
    "mathtext.fontset": "cm",
    "text.usetex": False,
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
    "lines.linewidth": 1.8,
    "lines.markersize": 5,
})


from numpy.polynomial.legendre import legvander, Legendre
from scipy.linalg import qr
from pymor.algorithms.pod import pod
from pymor.vectorarrays.numpy import NumpyVectorSpace
from pymor.operators.numpy import NumpyMatrixOperator


# ============================================================
# Parameter domain
# ============================================================

class ParameterDomain:
    def __init__(self, mu1_range, mu2_range):
        self.mu1_min, self.mu1_max = mu1_range
        self.mu2_min, self.mu2_max = mu2_range

    def normalize(self, mu):
        mu = np.asarray(mu, dtype=float)
        mu1 = mu[..., 0]
        mu2 = mu[..., 1]

        m1 = 2.0 * (mu1 - self.mu1_min) / (self.mu1_max - self.mu1_min) - 1.0
        m2 = 2.0 * (mu2 - self.mu2_min) / (self.mu2_max - self.mu2_min) - 1.0
        return m1, m2

    def denormalize(self, m1, m2):
        m1 = np.asarray(m1, dtype=float)
        m2 = np.asarray(m2, dtype=float)
        mu1 = 0.5 * (m1 + 1.0) * (self.mu1_max - self.mu1_min) + self.mu1_min
        mu2 = 0.5 * (m2 + 1.0) * (self.mu2_max - self.mu2_min) + self.mu2_min
        return np.column_stack((mu1, mu2))

    def random_samples(self, n, seed=0):
        rng = np.random.default_rng(seed)
        mu1 = rng.uniform(self.mu1_min, self.mu1_max, n)
        mu2 = rng.uniform(self.mu2_min, self.mu2_max, n)
        return np.column_stack((mu1, mu2))


# ============================================================
# Geometry deformation
# ============================================================

class TopRightDeformation:
    def evaluate(self, mu):
        mu1, mu2 = mu
        return CoefficientFunction((x * y * mu1, x * y * mu2))


class GeometryHandler:
    def __init__(self, mesh, order, deformation_model):
        self.mesh = mesh
        self.deformation_model = deformation_model
        self.deform_gf = GridFunction(VectorH1(mesh, order=order))

    def apply(self, mu):
        self.deform_gf.Set(self.deformation_model.evaluate(mu))
        self.mesh.SetDeformation(self.deform_gf)

    def clear(self):
        self.mesh.UnsetDeformation()


# ============================================================
# Darcy model
# ============================================================

class DarcyModel:
    def __init__(self, K=1.0):
        self.K = float(K)
        self.Kinv = 1.0 / self.K

    def p_exact(self):
        return x * x - y * y

    def u_exact(self):
        return CoefficientFunction((-2.0 * self.K * x, 2.0 * self.K * y))


# ============================================================
# Parameter sampling
# ============================================================

def sample_parameters(method, N, domain: ParameterDomain, seed=0):
    mu1_min, mu1_max = domain.mu1_min, domain.mu1_max
    mu2_min, mu2_max = domain.mu2_min, domain.mu2_max

    if method == "grid":
        mu1 = np.linspace(mu1_min, mu1_max, N)
        mu2 = np.linspace(mu2_min, mu2_max, N)
        mu = np.array([(a, b) for a in mu1 for b in mu2], dtype=float)

        w = np.ones((N, N))
        w[0, 1:-1] = w[-1, 1:-1] = w[1:-1, 0] = w[1:-1, -1] = 0.5
        w[0, 0] = w[0, -1] = w[-1, 0] = w[-1, -1] = 0.25
        return mu, np.diag(w.flatten())

    if method == "random":
        rng = np.random.default_rng(seed)
        mu1 = rng.uniform(mu1_min, mu1_max, N * N)
        mu2 = rng.uniform(mu2_min, mu2_max, N * N)
        return np.column_stack((mu1, mu2)), np.eye(N * N)

    raise ValueError(f"Unknown sampling method: {method}")


# ============================================================
# Legendre basis on parameter domain
# ============================================================

def legendre_basis(mu, P, domain: ParameterDomain):
    mu = np.asarray(mu, dtype=float)
    if mu.ndim == 1:
        mu = mu[None, :]

    m1, m2 = domain.normalize(mu)

    Vx = legvander(m1, P)
    Vy = legvander(m2, P)

    B = (Vx[:, :, None] * Vy[:, None, :]).reshape(len(mu), -1)
    return B[0] if len(mu) == 1 else B


# ============================================================
# POD helpers
# ============================================================

def build_pod_basis_from_snapshots(snapshots, product_mat):
    nsnaps, ndof = snapshots.shape
    space = NumpyVectorSpace(ndof)
    S = space.make_array(snapshots)
    product = NumpyMatrixOperator(product_mat)
    RB, svals = pod(S, rtol=1e-16, product=product)
    return RB.to_numpy().T, np.asarray(svals, dtype=float)


def choose_rank_from_svals(svals, tau):
    svals = np.asarray(svals, dtype=float)
    if len(svals) == 0:
        return 0
    if len(svals) == 1:
        return 1

    s2 = svals ** 2
    total = np.sum(s2)
    if total <= 0:
        return 1

    tail = np.sqrt(np.cumsum(s2[::-1])[::-1] / total)[1:]
    idx = np.where(tail < tau)[0]
    return int(idx[0] + 1) if len(idx) else len(svals)


def discarded_energy_curve(svals):
    """
    Returns:
        r_vals: 0,1,...,N
        disc[r] = sum_{j=r}^{N-1} sigma_j^2 / sum_{j=0}^{N-1} sigma_j^2
    """
    svals = np.asarray(svals, dtype=float)
    N = len(svals)

    if N == 0:
        return np.array([0]), np.array([0.0])

    s2 = svals ** 2
    total = np.sum(s2)

    if total <= 0:
        return np.arange(N + 1), np.zeros(N + 1)

    tail = np.zeros(N + 1, dtype=float)
    tail[:N] = np.sqrt(np.cumsum(s2[::-1])[::-1] / total)
    tail[N] = 0.0

    return np.arange(N + 1), tail


# ============================================================
# Weighted metric orthonormalization
# ============================================================

def metric_orthonormalize(V, M, tol=1e-12):
    if V.size == 0:
        return V

    G = V.T @ M @ V
    G = 0.5 * (G + G.T)

    evals, evecs = np.linalg.eigh(G)
    keep = evals > tol * max(1.0, np.max(np.abs(evals)))
    if not np.any(keep):
        return np.zeros((V.shape[0], 0))

    L = evecs[:, keep] @ np.diag(1.0 / np.sqrt(evals[keep]))
    Qm = V @ L
    return Qm


# ============================================================
# Weighted least-squares Legendre fit
# ============================================================

def weighted_legendre_fit(P, mu_train, arrays, W, domain, tol=1e-12):
    """
    Fit:
        array(mu) ~ sum_q theta_q(mu) * coeff_q
    arrays shape: (m, n1, n2, ...) or (m, n)
    returns coeffs shape: (Q, n1, n2, ...) or (Q, n)
    """
    B = np.array([legendre_basis(mu, P, domain) for mu in mu_train], dtype=float)
    m, Q = B.shape

    w = np.sqrt(np.diag(W))
    Bw = B * w[:, None]

    arrays = np.asarray(arrays, dtype=float)
    out_shape = arrays.shape[1:]
    arrays_flat = arrays.reshape(m, -1) * w[:, None]

    Qw, R, piv = qr(Bw, mode="economic", pivoting=True)

    diagR = np.abs(np.diag(R))
    thresh = tol * np.max(diagR) if len(diagR) else tol
    rank = np.sum(diagR > thresh)

    coeffs_flat = np.zeros((Q, arrays_flat.shape[1]), dtype=float)

    if rank > 0:
        Qr = Qw[:, :rank]
        Rr = R[:rank, :rank]
        pivr = piv[:rank]
        coeffs_r = np.linalg.solve(Rr, Qr.T @ arrays_flat)
        coeffs_flat[pivr, :] = coeffs_r

    return coeffs_flat.reshape((Q,) + out_shape)


def evaluate_legendre_fit(mu, P, coeffs, domain):
    theta = np.asarray(legendre_basis(mu, P, domain), dtype=float)
    return np.tensordot(theta, coeffs, axes=(0, 0))


def robust_linear_solve(A, b):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    try:
        return np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        # Fallback for rank-deficient fitted operators at some parameter points.
        x, *_ = np.linalg.lstsq(A, b, rcond=1e-12)
        return x


# ============================================================
# Gauss-Lobatto quadrature on parameter space
# ============================================================

def gauss_lobatto_rule_1d(n):
    if n < 2:
        raise ValueError("Gauss-Lobatto needs at least n=2 points.")

    if n == 2:
        xq = np.array([-1.0, 1.0])
        wq = np.array([1.0, 1.0])
        return xq, wq

    P = Legendre.basis(n - 1)
    dP = P.deriv()
    interior = np.sort(dP.roots())

    xq = np.hstack(([-1.0], interior, [1.0]))
    wq = np.empty(n, dtype=float)
    wq[0] = wq[-1] = 2.0 / (n * (n - 1))

    vals = P(interior)
    wq[1:-1] = 2.0 / (n * (n - 1) * vals ** 2)
    return xq, wq


def tensor_gauss_lobatto_parameter_rule(domain: ParameterDomain, n1d):
    x1, w1 = gauss_lobatto_rule_1d(n1d)
    x2, w2 = gauss_lobatto_rule_1d(n1d)

    X1, X2 = np.meshgrid(x1, x2, indexing="ij")
    W = np.outer(w1, w2)

    mus = domain.denormalize(X1.ravel(), X2.ravel())

    J1 = 0.5 * (domain.mu1_max - domain.mu1_min)
    J2 = 0.5 * (domain.mu2_max - domain.mu2_min)
    weights = (J1 * J2) * W.ravel()

    return mus, weights


# ============================================================
# Metric-based field errors
# ============================================================

def build_div_product(Mp, B):
    Mdiv = B.T @ np.linalg.solve(Mp, B)
    return 0.5 * (Mdiv + Mdiv.T)


def build_hdiv_product(Mu, Mp, B):
    return Mu + build_div_product(Mp, B)


def metric_norm(x, M, eps=1e-14):
    x = np.asarray(x, dtype=float)
    M = np.asarray(M, dtype=float)
    val = float(x.T @ M @ x)
    return np.sqrt(max(val, 0.0))


def squared_metric_norm(x, M):
    x = np.asarray(x, dtype=float)
    M = np.asarray(M, dtype=float)
    val = float(x.T @ M @ x)
    return max(val, 0.0)


def relative_metric_error(x_approx, x_true, M, eps=1e-14):
    e = np.asarray(x_approx, dtype=float) - np.asarray(x_true, dtype=float)
    num = metric_norm(e, M, eps=eps)
    den = max(metric_norm(x_true, M, eps=eps), eps)
    return num / den


def hdiv_error_divergence_share(x_approx, x_true, Mu, Mdiv):
    e = np.asarray(x_approx, dtype=float) - np.asarray(x_true, dtype=float)
    l2_part = squared_metric_norm(e, Mu)
    div_part = squared_metric_norm(e, Mdiv)
    return div_part / max(l2_part + div_part, 1e-30)


# ============================================================
# Mixed Darcy FOM solver
# ============================================================

class MixedDarcyFOMSolver:
    def __init__(self, mesh, order, darcy_model: DarcyModel):
        self.mesh = mesh
        self.order = order
        self.model = darcy_model

    def solve(self, collect_system=True):
        V = HDiv(self.mesh, order=self.order)
        Q = L2(self.mesh, order=self.order - 1)
        Y = FESpace([V, Q])

        (u, p) = Y.TrialFunction()
        (v, q) = Y.TestFunction()
        n = specialcf.normal(self.mesh.dim)

        # full mixed system
        a = BilinearForm(Y, symmetric=False)
        a += (
            self.model.Kinv * InnerProduct(u, v)
            - p * div(v)
            + q * div(u)
        ) * dx

        L = LinearForm(Y)
        L += (-self.model.p_exact() * InnerProduct(v.Trace(), n)) * ds

        gfu = GridFunction(Y)
        a.Assemble()
        L.Assemble()

        inv = a.mat.Inverse(Y.FreeDofs(), inverse="pardiso")
        gfu.vec.data = inv * L.vec

        uh, ph = gfu.components

        out = {
            "gfu": gfu,
            "V": V,
            "Q": Q,
            "Y": Y,
            "Nu": V.ndof,
            "Np": Q.ndof,
            "u": uh.vec.FV().NumPy().copy(),
            "p": ph.vec.FV().NumPy().copy(),
            "full": gfu.vec.FV().NumPy().copy(),
        }

        if not collect_system:
            return out

        # Only assemble and export dense operators when study utilities need them.
        uV, vV = V.TrialFunction(), V.TestFunction()
        a_uu = BilinearForm(V, symmetric=True)
        a_uu += self.model.Kinv * InnerProduct(uV, vV) * dx
        a_uu.Assemble()

        pQ, qQ = Q.TrialFunction(), Q.TestFunction()
        mp = BilinearForm(Q, symmetric=True)
        mp += pQ * qQ * dx
        mp.Assemble()

        b = BilinearForm(trialspace=V, testspace=Q)
        b += qQ * div(uV) * dx
        b.Assemble()

        out.update({
            "A": np.array(a.mat.ToDense(), dtype=float),
            "f": np.array(L.vec.FV().NumPy(), dtype=float),
            "Auu": np.array(a_uu.mat.ToDense(), dtype=float),
            "Mp": np.array(mp.mat.ToDense(), dtype=float),
            "B": np.array(b.mat.ToDense(), dtype=float),
        })
        return out


# ============================================================
# Experiment wrapper
# ============================================================

class DarcyROMExperiment:
    def __init__(self, mesh, order, darcy_model, domain, deformation_model):
        self.mesh = mesh
        self.order = order
        self.domain = domain

        self.geom = GeometryHandler(mesh, order, deformation_model)
        self.solver = MixedDarcyFOMSolver(mesh, order, darcy_model)

        self._reference_products_cache = None

    def solve_full(self, mu, collect_system=True):
        self.geom.apply(mu)
        out = self.solver.solve(collect_system=collect_system)
        self.geom.clear()
        return out

    def assemble_reference_products(self):
        V = HDiv(self.mesh, order=self.order)
        Q = L2(self.mesh, order=self.order - 1)

        u, v = V.TrialFunction(), V.TestFunction()
        p, q = Q.TrialFunction(), Q.TestFunction()

        Mu = BilinearForm(V, symmetric=True)
        Mu += self.solver.model.Kinv * InnerProduct(u, v) * dx
        Mu.Assemble()

        Mp = BilinearForm(Q, symmetric=True)
        Mp += p * q * dx
        Mp.Assemble()

        B = BilinearForm(trialspace=V, testspace=Q)
        B += q * div(u) * dx
        B.Assemble()

        return (
            np.array(Mu.mat.ToDense(), dtype=float),
            np.array(Mp.mat.ToDense(), dtype=float),
            np.array(B.mat.ToDense(), dtype=float),
            V.ndof,
            Q.ndof,
        )

    def get_reference_products(self):
        if self._reference_products_cache is None:
            Mu, Mp, B, Nu, Np = self.assemble_reference_products()
            Mdiv = build_div_product(Mp, B)
            Mhdiv = Mu + Mdiv
            self._reference_products_cache = {
                "Mu": Mu,
                "Mp": Mp,
                "B": B,
                "Mdiv": Mdiv,
                "Mhdiv": Mhdiv,
                "Nu": Nu,
                "Np": Np,
            }
        return self._reference_products_cache


# ============================================================
# ROM with supremizers
# ============================================================

class MixedDarcyL2FitROM:
    def __init__(self, experiment: DarcyROMExperiment):
        self.experiment = experiment
        self.domain = experiment.domain

        self.Nu = None
        self.Np = None

        self.Mu_ref = None
        self.Mp_ref = None
        self.B_ref = None
        self.Mhdiv_ref = None

        self.Vu_pod = None
        self.Vp = None
        self.S_sup = None
        self.Vu_stab = None
        self.Vrb = None

        self.r_u_pod = None
        self.r_p = None
        self.r_u_stab = None
        self.r_tot = None

        self.svals_u = None
        self.svals_p = None

        self.P = None
        self.A_tilde_r = None
        self.f_tilde_r = None

    def set_training_data(self, Uu, Up, As, fs, Nu, Np):
        self.Uu = np.asarray(Uu, dtype=float)
        self.Up = np.asarray(Up, dtype=float)
        self.As = np.asarray(As, dtype=float)
        self.fs = np.asarray(fs, dtype=float)
        self.Nu = int(Nu)
        self.Np = int(Np)

    def build_pod_bases(self, tau_u=1e-8, tau_p=1e-8):
        self.Mu_ref, self.Mp_ref, self.B_ref, Nu_ref, Np_ref = self.experiment.assemble_reference_products()
        self.Mhdiv_ref = build_hdiv_product(self.Mu_ref, self.Mp_ref, self.B_ref)

        if self.Nu is None:
            self.Nu = Nu_ref
            self.Np = Np_ref

        Vu_full, svals_u = build_pod_basis_from_snapshots(self.Uu, product_mat=self.Mu_ref)
        Vp_full, svals_p = build_pod_basis_from_snapshots(self.Up, product_mat=self.Mp_ref)

        self.svals_u = svals_u
        self.svals_p = svals_p

        self.r_u_pod = choose_rank_from_svals(svals_u, tau_u)
        self.r_p = choose_rank_from_svals(svals_p, tau_p)

        self.Vu_pod = Vu_full[:, :self.r_u_pod]
        self.Vp = Vp_full[:, :self.r_p]

        return {
            "Vu_full": Vu_full,
            "Vp_full": Vp_full,
            "svals_u": svals_u,
            "svals_p": svals_p,
            "r_u_pod": self.r_u_pod,
            "r_p": self.r_p,
        }

    def build_supremizers(self):
        if self.Vp is None:
            raise RuntimeError("Build pressure POD basis before supremizers.")

        rhs = self.B_ref.T @ self.Vp
        S = np.linalg.solve(self.Mu_ref, rhs)

        self.S_sup = S

        Vaug = np.column_stack([self.Vu_pod, self.S_sup])
        self.Vu_stab = metric_orthonormalize(Vaug, self.Mu_ref)

        self.r_u_stab = self.Vu_stab.shape[1]
        self.r_tot = self.r_u_stab + self.r_p

        self.Vrb = np.block([
            [self.Vu_stab, np.zeros((self.Nu, self.r_p))],
            [np.zeros((self.Np, self.r_u_stab)), self.Vp]
        ])

        return {
            "S_sup": self.S_sup,
            "Vu_stab": self.Vu_stab,
            "r_u_stab": self.r_u_stab,
            "r_tot": self.r_tot,
        }

    def fit_from_training_data(self, mu_train, W_train, P=4, tau_u=1e-8, tau_p=1e-8):
        self.P = int(P)

        pod_data = self.build_pod_bases(tau_u=tau_u, tau_p=tau_p)
        self.build_supremizers()

        m = len(mu_train)
        Ars = np.empty((m, self.r_tot, self.r_tot), dtype=float)
        frs = np.empty((m, self.r_tot), dtype=float)

        for i in range(m):
            Ars[i] = self.Vrb.T @ self.As[i] @ self.Vrb
            frs[i] = self.Vrb.T @ self.fs[i]

        self.A_tilde_r = weighted_legendre_fit(
            P=self.P,
            mu_train=mu_train,
            arrays=Ars,
            W=W_train,
            domain=self.domain,
        )
        self.f_tilde_r = weighted_legendre_fit(
            P=self.P,
            mu_train=mu_train,
            arrays=frs,
            W=W_train,
            domain=self.domain,
        )

        return {
            "P": self.P,
            "Vu_pod": self.Vu_pod,
            "Vp": self.Vp,
            "S_sup": self.S_sup,
            "Vu_stab": self.Vu_stab,
            "Vrb": self.Vrb,
            "r_u_pod": self.r_u_pod,
            "r_p": self.r_p,
            "r_u_stab": self.r_u_stab,
            "r_tot": self.r_tot,
            "A_tilde_r": self.A_tilde_r,
            "f_tilde_r": self.f_tilde_r,
            "svals_u": self.svals_u,
            "svals_p": self.svals_p,
            "pod_data": pod_data,
        }

    def solve_online(self, mu, reconstruct_full=True):
        if self.A_tilde_r is None:
            raise RuntimeError("Fit the ROM before calling solve_online.")

        A_mu = evaluate_legendre_fit(mu, self.P, self.A_tilde_r, self.domain)
        f_mu = evaluate_legendre_fit(mu, self.P, self.f_tilde_r, self.domain)

        z = np.linalg.solve(A_mu, f_mu)
        if reconstruct_full:
            full = self.Vrb @ z
            u_rom = full[:self.Nu]
            p_rom = full[self.Nu:]
        else:
            full = np.array([], dtype=float)
            u_rom = np.array([], dtype=float)
            p_rom = np.array([], dtype=float)

        return {
            "z": z,
            "u": u_rom,
            "p": p_rom,
            "full": full,
            "A_r": A_mu,
            "f_r": f_mu,
        }


# ============================================================
# Study helpers
# ============================================================

def collect_training_data(experiment, mu_train):
    Uu, Up, As, fs = [], [], [], []
    Nu = None
    Np = None

    for mu in mu_train:
        out = experiment.solve_full(tuple(mu))
        Uu.append(out["u"])
        Up.append(out["p"])
        As.append(out["A"])
        fs.append(out["f"])

        if Nu is None:
            Nu = out["Nu"]
            Np = out["Np"]

    return (
        np.array(Uu, dtype=float),
        np.array(Up, dtype=float),
        np.array(As, dtype=float),
        np.array(fs, dtype=float),
        Nu,
        Np,
    )


def collect_snapshot_data(experiment, mu_train):
    """Collect only (u, p) snapshots to keep memory footprint low."""
    Uu, Up = [], []
    Nu = None
    Np = None

    for mu in mu_train:
        out = experiment.solve_full(tuple(mu), collect_system=False)
        Uu.append(out["u"])
        Up.append(out["p"])

        if Nu is None:
            Nu = out["Nu"]
            Np = out["Np"]

    return np.array(Uu, dtype=float), np.array(Up, dtype=float), int(Nu), int(Np)


def fit_rom_streaming(
    experiment,
    mu_train,
    W_train,
    Uu_train,
    Up_train,
    Nu,
    Np,
    P,
    tau,
):
    """
    Memory-efficient ROM fit:
    - Build POD/supremizer bases from snapshots.
    - Assemble reduced operators sample-by-sample (no full As_train/fs_train storage).
    """
    rom = MixedDarcyL2FitROM(experiment)
    rom.Uu = np.asarray(Uu_train, dtype=float)
    rom.Up = np.asarray(Up_train, dtype=float)
    rom.Nu = int(Nu)
    rom.Np = int(Np)

    rom.P = int(P)
    rom.build_pod_bases(tau_u=float(tau), tau_p=float(tau))
    rom.build_supremizers()

    m = len(mu_train)
    Ars = np.empty((m, rom.r_tot, rom.r_tot), dtype=float)
    frs = np.empty((m, rom.r_tot), dtype=float)

    for i, mu in enumerate(mu_train):
        out = experiment.solve_full(tuple(mu), collect_system=True)
        A = out["A"]
        f = out["f"]
        Ars[i] = rom.Vrb.T @ A @ rom.Vrb
        frs[i] = rom.Vrb.T @ f

    rom.A_tilde_r = weighted_legendre_fit(
        P=rom.P,
        mu_train=mu_train,
        arrays=Ars,
        W=W_train,
        domain=rom.domain,
    )
    rom.f_tilde_r = weighted_legendre_fit(
        P=rom.P,
        mu_train=mu_train,
        arrays=frs,
        W=W_train,
        domain=rom.domain,
    )

    return rom


def collect_test_solutions(experiment, mu_test):
    data = []
    for mu in mu_test:
        out = experiment.solve_full(tuple(mu))
        data.append({
            "mu": tuple(mu),
            "u": out["u"],
            "p": out["p"],
            "full": out["full"],
            "A": out["A"],
            "f": out["f"],
        })
    return data


def compute_l2_fit_matrix_error(experiment, P, mu_train, W_train, domain, gl_n1d):
    """
    Memory-efficient equivalent of the previous implementation.
    Computes L2-fit coefficients for full A(mu), f(mu) without storing all A snapshots.
    """
    mu_train = np.asarray(mu_train, dtype=float)
    m = len(mu_train)
    if m == 0:
        raise ValueError("mu_train is empty")

    B = np.array([legendre_basis(mu, P, domain) for mu in mu_train], dtype=float)
    _, Q = B.shape

    w_sqrt = np.sqrt(np.diag(W_train))
    Bw = B * w_sqrt[:, None]

    Qw, R, piv = qr(Bw, mode="economic", pivoting=True)
    diagR = np.abs(np.diag(R))
    thresh = 1e-12 * np.max(diagR) if len(diagR) else 1e-12
    rank = int(np.sum(diagR > thresh))

    # Discover full system size from first sample.
    first_out = experiment.solve_full(tuple(mu_train[0]))
    n_full = int(len(first_out["f"]))
    nA = n_full * n_full

    YA = np.zeros((rank, nA), dtype=float)
    Yf = np.zeros((rank, n_full), dtype=float)

    for i, mu in enumerate(mu_train):
        out = first_out if i == 0 else experiment.solve_full(tuple(mu))
        A_flat_w = np.asarray(out["A"], dtype=float).reshape(-1) * w_sqrt[i]
        f_w = np.asarray(out["f"], dtype=float) * w_sqrt[i]

        qrow = Qw[i, :rank]
        for r_idx, qr_val in enumerate(qrow):
            if qr_val != 0.0:
                YA[r_idx, :] += qr_val * A_flat_w
                Yf[r_idx, :] += qr_val * f_w

    coeffs_A_flat = np.zeros((Q, nA), dtype=float)
    coeffs_f = np.zeros((Q, n_full), dtype=float)

    if rank > 0:
        Rr = R[:rank, :rank]
        pivr = piv[:rank]
        XA = np.linalg.solve(Rr, YA)
        Xf = np.linalg.solve(Rr, Yf)
        coeffs_A_flat[pivr, :] = XA
        coeffs_f[pivr, :] = Xf

    A_tilde_full = coeffs_A_flat.reshape(Q, n_full, n_full)
    f_tilde_full = coeffs_f

    mu_quad, w_quad = tensor_gauss_lobatto_parameter_rule(domain, gl_n1d)

    num_A = 0.0
    den_A = 0.0
    max_A = 0.0

    for mu, w in zip(mu_quad, w_quad):
        out = experiment.solve_full(tuple(mu))
        A_true = out["A"]
        A_fit = evaluate_legendre_fit(mu, P, A_tilde_full, domain)

        diff = np.linalg.norm(A_fit - A_true, ord="fro")
        ref = max(np.linalg.norm(A_true, ord="fro"), 1e-30)

        num_A += w * diff**2
        den_A += w * ref**2
        max_A = max(max_A, diff / ref)

    rel_A_int = np.sqrt(num_A / max(den_A, 1e-30))
    return rel_A_int, max_A, A_tilde_full, f_tilde_full

def compute_l2_fit_solution_errors_integrated_and_max(
    experiment, P, A_tilde_full, f_tilde_full, domain, Mu, Mp, Mhdiv, gl_n1d, Mdiv=None
):
    mu_quad, w_quad = tensor_gauss_lobatto_parameter_rule(domain, gl_n1d)

    num_u_l2 = 0.0
    den_u_l2 = 0.0

    num_u_hdiv = 0.0
    den_u_hdiv = 0.0

    num_p_l2 = 0.0
    den_p_l2 = 0.0

    max_u_l2 = 0.0
    max_u_hdiv = 0.0
    max_p_l2 = 0.0
    div_shares = []

    for mu, w in zip(mu_quad, w_quad):
        out = experiment.solve_full(tuple(mu))
        u_true = out["u"]
        p_true = out["p"]

        A_fit = evaluate_legendre_fit(mu, P, A_tilde_full, domain)
        f_fit = evaluate_legendre_fit(mu, P, f_tilde_full, domain)
        full_fit = robust_linear_solve(A_fit, f_fit)

        Nu = len(u_true)
        u_fit = full_fit[:Nu]
        p_fit = full_fit[Nu:]

        eu = u_fit - u_true
        ep = p_fit - p_true

        num_u_l2 += w * squared_metric_norm(eu, Mu)
        den_u_l2 += w * squared_metric_norm(u_true, Mu)

        num_u_hdiv += w * squared_metric_norm(eu, Mhdiv)
        den_u_hdiv += w * squared_metric_norm(u_true, Mhdiv)

        num_p_l2 += w * squared_metric_norm(ep, Mp)
        den_p_l2 += w * squared_metric_norm(p_true, Mp)

        err_u_l2 = relative_metric_error(u_fit, u_true, Mu)
        err_u_hdiv = relative_metric_error(u_fit, u_true, Mhdiv)
        err_p_l2 = relative_metric_error(p_fit, p_true, Mp)
        if Mdiv is not None:
            div_shares.append(hdiv_error_divergence_share(u_fit, u_true, Mu, Mdiv))

        max_u_l2 = max(max_u_l2, err_u_l2)
        max_u_hdiv = max(max_u_hdiv, err_u_hdiv)
        max_p_l2 = max(max_p_l2, err_p_l2)

    return {
        "u_l2_int": np.sqrt(num_u_l2 / max(den_u_l2, 1e-30)),
        "u_hdiv_int": np.sqrt(num_u_hdiv / max(den_u_hdiv, 1e-30)),
        "p_l2_int": np.sqrt(num_p_l2 / max(den_p_l2, 1e-30)),
        "u_l2_max": max_u_l2,
        "u_hdiv_max": max_u_hdiv,
        "p_l2_max": max_p_l2,
        "u_hdiv_div_share_mean": float(np.mean(div_shares)) if div_shares else np.nan,
        "u_hdiv_div_share_max": float(np.max(div_shares)) if div_shares else np.nan,
    }


def compute_rom_solution_errors(
    experiment,
    Uu_train,
    Up_train,
    Nu,
    Np,
    mu_train,
    W_train,
    P,
    tau,
    test_data,
    Mu,
    Mp,
    Mhdiv,
    Mdiv=None,
    timing_mus=None,
    timing_repeats=0,
):
    rom = fit_rom_streaming(
        experiment=experiment,
        mu_train=mu_train,
        W_train=W_train,
        Uu_train=Uu_train,
        Up_train=Up_train,
        Nu=Nu,
        Np=Np,
        P=P,
        tau=tau,
    )

    rom_data = {
        "r_tot": int(rom.r_tot),
            "r_triplet": r_triplet_from_stabilized_dims(rom.r_u_stab, rom.r_p),
        "svals_u": np.array(rom.svals_u, dtype=float),
        "svals_p": np.array(rom.svals_p, dtype=float),
    }

    errs_u_l2 = []
    errs_u_hdiv = []
    errs_p_l2 = []
    div_shares = []

    for item in test_data:
        mu = item["mu"]
        u_true = item["u"]
        p_true = item["p"]

        rom_sol = rom.solve_online(mu)
        u_rom = rom_sol["u"]
        p_rom = rom_sol["p"]

        errs_u_l2.append(relative_metric_error(u_rom, u_true, Mu))
        errs_u_hdiv.append(relative_metric_error(u_rom, u_true, Mhdiv))
        errs_p_l2.append(relative_metric_error(p_rom, p_true, Mp))
        if Mdiv is not None:
            div_shares.append(hdiv_error_divergence_share(u_rom, u_true, Mu, Mdiv))

    timing = None
    if timing_mus is not None and timing_repeats > 0 and len(timing_mus) > 0:
        rom.solve_online(timing_mus[0], reconstruct_full=False)
        rom_times = []
        for _ in range(timing_repeats):
            t0 = time.perf_counter()
            for mu in timing_mus:
                rom.solve_online(mu, reconstruct_full=False)
            rom_times.append(time.perf_counter() - t0)

        rom_total_mean = float(np.mean(rom_times))
        timing = {
            "rom_times": np.array(rom_times, dtype=float),
            "rom_total_mean": rom_total_mean,
            "rom_time_per_solve": rom_total_mean / len(timing_mus),
        }

    return {
        "u_l2_mean": float(np.mean(errs_u_l2)),
        "u_l2_max": float(np.max(errs_u_l2)),
        "u_hdiv_mean": float(np.mean(errs_u_hdiv)),
        "u_hdiv_max": float(np.max(errs_u_hdiv)),
        "u_hdiv_div_share_mean": float(np.mean(div_shares)) if div_shares else np.nan,
        "u_hdiv_div_share_max": float(np.max(div_shares)) if div_shares else np.nan,
        "p_l2_mean": float(np.mean(errs_p_l2)),
        "p_l2_max": float(np.max(errs_p_l2)),
    }, rom_data, timing

def benchmark_rom_speedup_vs_polynomial_degree(
    experiment,
    domain,
    mu_train,
    W_train,
    P_values,
    tau_values,
    n_test=40,
    random_seed=1234,
    n_repeats=3,
):
    Uu_train, Up_train, Nu, Np = collect_snapshot_data(experiment, mu_train)

    mu_test = domain.random_samples(n_test, seed=random_seed)
    mu_test = [tuple(mu) for mu in mu_test]

    # Warm up the FOM path once before timing.
    experiment.solve_full(mu_test[0], collect_system=False)

    fom_times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        for mu in mu_test:
            experiment.solve_full(mu, collect_system=False)
        fom_times.append(time.perf_counter() - t0)

    fom_total_mean = float(np.mean(fom_times))
    nsamples = len(mu_test)

    speedup_by_tau = {tau: [] for tau in tau_values}
    rom_modes_by_tau = {tau: [] for tau in tau_values}
    rom_total_by_tau = {tau: [] for tau in tau_values}
    rom_per_solve_by_tau = {tau: [] for tau in tau_values}

    for P in P_values:
        for tau in tau_values:
            rom = fit_rom_streaming(
                experiment=experiment,
                mu_train=mu_train,
                W_train=W_train,
                Uu_train=Uu_train,
                Up_train=Up_train,
                Nu=Nu,
                Np=Np,
                P=P,
                tau=tau,
            )

            rom.solve_online(mu_test[0], reconstruct_full=True)

            rom_times = []
            for _ in range(n_repeats):
                t0 = time.perf_counter()
                for mu in mu_test:
                    rom.solve_online(mu, reconstruct_full=True)
                rom_times.append(time.perf_counter() - t0)

            rom_total_mean = float(np.mean(rom_times))
            speedup_by_tau[tau].append(fom_total_mean / rom_total_mean)
            rom_modes_by_tau[tau].append(int(rom.r_tot))
            rom_total_by_tau[tau].append(rom_total_mean)
            rom_per_solve_by_tau[tau].append(rom_total_mean / nsamples)

    return {
        "P_values": np.array(P_values, dtype=int),
        "tau_values": np.array(tau_values, dtype=float),
        "speedup_by_tau": {
            float(tau): np.array(values, dtype=float)
            for tau, values in speedup_by_tau.items()
        },
        "rom_modes_by_tau": {
            float(tau): np.array(values, dtype=int)
            for tau, values in rom_modes_by_tau.items()
        },
        "rom_total_by_tau": {
            float(tau): np.array(values, dtype=float)
            for tau, values in rom_total_by_tau.items()
        },
        "rom_per_solve_by_tau": {
            float(tau): np.array(values, dtype=float)
            for tau, values in rom_per_solve_by_tau.items()
        },
        "n_test": nsamples,
        "n_repeats": int(n_repeats),
        "mu_test": np.array(mu_test, dtype=float),
        "fom_times": np.array(fom_times, dtype=float),
        "fom_total_mean": fom_total_mean,
        "fom_time_per_solve": fom_total_mean / nsamples,
    }

def _json_ready(obj):
    if isinstance(obj, dict):
        return {str(k): _json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_ready(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def save_results_bundle_json(path, results=None, speed_results=None, extra=None):
    payload = {
        "results": _json_ready(results),
        "speed_results": _json_ready(speed_results),
        "extra": _json_ready(extra) if extra is not None else {},
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def load_results_bundle_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def speed_results_from_convergence(results):
    if "speed" not in results:
        raise ValueError("Convergence results do not contain speed data. Run with measure_speed=True.")

    tau_values = np.asarray(results["tau_values"], dtype=float)
    P_values = np.asarray(results["P_values"], dtype=int)

    return {
        "P_values": P_values,
        "tau_values": tau_values,
        "speedup_by_tau": {
            float(tau): np.asarray(results["speed"]["speedup_by_tau"][tau], dtype=float)
            for tau in tau_values
        },
        "rom_modes_by_tau": {
            float(tau): np.full(len(P_values), int(results["rom_modes"][tau]), dtype=int)
            for tau in tau_values
        },
        "rom_total_by_tau": {
            float(tau): np.asarray(results["speed"]["rom_total_by_tau"][tau], dtype=float)
            for tau in tau_values
        },
        "rom_per_solve_by_tau": {
            float(tau): np.asarray(results["speed"]["rom_per_solve_by_tau"][tau], dtype=float)
            for tau in tau_values
        },
        "n_test": int(results["speed"]["n_test"]),
        "n_repeats": int(results["speed"]["n_repeats"]),
        "mu_test": np.asarray(results["speed"]["mu_test"], dtype=float),
        "fom_times": np.asarray(results["speed"]["fom_times"], dtype=float),
        "fom_total_mean": float(results["speed"]["fom_total_mean"]),
        "fom_time_per_solve": float(results["speed"]["fom_time_per_solve"]),
    }

def run_convergence_study(
    experiment,
    domain,
    P_values,
    tau_values,
    mu_train,
    W_train,
    n_test_random=40,
    random_seed=1234,
    gl_n1d=8,
    measure_speed=False,
    speed_repeats=3,
):
    Uu_train, Up_train, Nu, Np = collect_snapshot_data(experiment, mu_train)

    mu_test = domain.random_samples(n_test_random, seed=random_seed)
    test_data = collect_test_solutions(experiment, mu_test)

    ref_products = experiment.get_reference_products()
    Mu = ref_products["Mu"]
    Mp = ref_products["Mp"]
    Mdiv = ref_products["Mdiv"]
    Mhdiv = ref_products["Mhdiv"]
    div_to_l2 = np.linalg.norm(Mdiv, ord="fro") / max(np.linalg.norm(Mu, ord="fro"), 1e-30)
    print(f"H(div) diagnostic: ||Mdiv||_F / ||Mu||_F = {div_to_l2:.3e}")

    results = {
        "P_values": np.array(P_values, dtype=int),
        "tau_values": list(tau_values),
        "l2fit_operator_int": [],
        "l2fit_operator_max": [],
        "l2fit_u_l2_int": [],
        "l2fit_u_l2_max": [],
        "l2fit_u_hdiv_int": [],
        "l2fit_u_hdiv_max": [],
        "l2fit_u_hdiv_div_share_mean": [],
        "l2fit_u_hdiv_div_share_max": [],
        "l2fit_p_l2_int": [],
        "l2fit_p_l2_max": [],
        "hdiv_diagnostic_div_to_l2_fro": float(div_to_l2),
        "rom_errors": {
            tau: {
                "u_l2_mean": [],
                "u_l2_max": [],
                "u_hdiv_mean": [],
                "u_hdiv_max": [],
                "u_hdiv_div_share_mean": [],
                "u_hdiv_div_share_max": [],
                "p_l2_mean": [],
                "p_l2_max": [],
            }
            for tau in tau_values
        },
        "rom_modes": {},
        "svals_u": None,
        "svals_p": None,
    }

    timing_mus = [tuple(mu) for mu in mu_test]

    if measure_speed:
        speed = {
            "n_test": len(timing_mus),
            "n_repeats": int(speed_repeats),
            "mu_test": np.array(mu_test, dtype=float),
            "rom_total_by_tau": {tau: [] for tau in tau_values},
            "rom_per_solve_by_tau": {tau: [] for tau in tau_values},
            "speedup_by_tau": {tau: [] for tau in tau_values},
        }

        experiment.solve_full(timing_mus[0], collect_system=False)
        fom_times = []
        for _ in range(speed_repeats):
            t0 = time.perf_counter()
            for mu in timing_mus:
                experiment.solve_full(mu, collect_system=False)
            fom_times.append(time.perf_counter() - t0)

        speed["fom_times"] = np.array(fom_times, dtype=float)
        speed["fom_total_mean"] = float(np.mean(fom_times))
        speed["fom_time_per_solve"] = speed["fom_total_mean"] / max(1, len(timing_mus))
        results["speed"] = speed

    for P in P_values:
        Aerr_int, Aerr_max, A_tilde_full, f_tilde_full = compute_l2_fit_matrix_error(
            experiment=experiment,
            P=P,
            mu_train=mu_train,
            W_train=W_train,
            domain=domain,
            gl_n1d=gl_n1d,
        )

        fit_sol_errs = compute_l2_fit_solution_errors_integrated_and_max(
            experiment=experiment,
            P=P,
            A_tilde_full=A_tilde_full,
            f_tilde_full=f_tilde_full,
            domain=domain,
            Mu=Mu,
            Mp=Mp,
            Mhdiv=Mhdiv,
            gl_n1d=gl_n1d,
            Mdiv=Mdiv,
        )

        results["l2fit_operator_int"].append(Aerr_int)
        results["l2fit_operator_max"].append(Aerr_max)
        results["l2fit_u_l2_int"].append(fit_sol_errs["u_l2_int"])
        results["l2fit_u_l2_max"].append(fit_sol_errs["u_l2_max"])
        results["l2fit_u_hdiv_int"].append(fit_sol_errs["u_hdiv_int"])
        results["l2fit_u_hdiv_max"].append(fit_sol_errs["u_hdiv_max"])
        results["l2fit_u_hdiv_div_share_mean"].append(fit_sol_errs["u_hdiv_div_share_mean"])
        results["l2fit_u_hdiv_div_share_max"].append(fit_sol_errs["u_hdiv_div_share_max"])
        results["l2fit_p_l2_int"].append(fit_sol_errs["p_l2_int"])
        results["l2fit_p_l2_max"].append(fit_sol_errs["p_l2_max"])

        for tau in tau_values:
            rom_errs, rom_data, rom_timing = compute_rom_solution_errors(
                experiment=experiment,
                Uu_train=Uu_train,
                Up_train=Up_train,
                Nu=Nu,
                Np=Np,
                mu_train=mu_train,
                W_train=W_train,
                P=P,
                tau=tau,
                test_data=test_data,
                Mu=Mu,
                Mp=Mp,
                Mhdiv=Mhdiv,
                Mdiv=Mdiv,
                timing_mus=timing_mus if measure_speed else None,
                timing_repeats=speed_repeats if measure_speed else 0,
            )

            for key in rom_errs:
                results["rom_errors"][tau][key].append(rom_errs[key])

            if tau not in results["rom_modes"]:
                results["rom_modes"][tau] = rom_data["r_tot"]
                if "rom_r_components" not in results:
                    results["rom_r_components"] = {}
                r_p, r_u, r_s = rom_data.get("r_triplet", (np.nan, np.nan, np.nan))
                results["rom_r_components"][tau] = {
                    "r_p": float(r_p),
                    "r_s": float(r_s),
                    "r_u": float(r_u),
                }

            if results["svals_u"] is None:
                results["svals_u"] = np.array(rom_data["svals_u"], dtype=float)
                results["svals_p"] = np.array(rom_data["svals_p"], dtype=float)

            if measure_speed and rom_timing is not None:
                results["speed"]["rom_total_by_tau"][tau].append(rom_timing["rom_total_mean"])
                results["speed"]["rom_per_solve_by_tau"][tau].append(rom_timing["rom_time_per_solve"])
                results["speed"]["speedup_by_tau"][tau].append(
                    results["speed"]["fom_total_mean"] / rom_timing["rom_total_mean"]
                )

    for key in [
        "l2fit_operator_int",
        "l2fit_operator_max",
        "l2fit_u_l2_int",
        "l2fit_u_l2_max",
        "l2fit_u_hdiv_int",
        "l2fit_u_hdiv_max",
        "l2fit_u_hdiv_div_share_mean",
        "l2fit_u_hdiv_div_share_max",
        "l2fit_p_l2_int",
        "l2fit_p_l2_max",
    ]:
        results[key] = np.array(results[key], dtype=float)

    for tau in tau_values:
        for key in results["rom_errors"][tau]:
            results["rom_errors"][tau][key] = np.array(results["rom_errors"][tau][key], dtype=float)

    if measure_speed:
        for tau in tau_values:
            results["speed"]["rom_total_by_tau"][tau] = np.array(results["speed"]["rom_total_by_tau"][tau], dtype=float)
            results["speed"]["rom_per_solve_by_tau"][tau] = np.array(results["speed"]["rom_per_solve_by_tau"][tau], dtype=float)
            results["speed"]["speedup_by_tau"][tau] = np.array(results["speed"]["speedup_by_tau"][tau], dtype=float)

    return results

def plot_rom_velocity_errors(results, filename="rom_velocity_errors.pdf"):
    P_values = results["P_values"]
    tau_values = results["tau_values"]

    fig, ax = plt.subplots(figsize=(8, 5))

    tau_handles = []
    meta_handles = [
        Line2D([0], [0], color="black", marker="x", linestyle="None", markersize=7, label=r"$L^2$"),
        Line2D([0], [0], color="black", marker="s", markerfacecolor="none", linestyle="None", markersize=6, label=r"$H(\mathrm{div})$"),
        Line2D([0], [0], color="black", linestyle="-", linewidth=1.8, label="Mean"),
        Line2D([0], [0], color="black", linestyle="--", linewidth=1.8, label="Max"),
    ]

    for k, tau in enumerate(tau_values):
        rom_errs_map = results["rom_errors"]
        rom_modes_map = results["rom_modes"]
        errs = rom_errs_map.get(tau, rom_errs_map.get(str(tau)))
        r_tot = rom_modes_map.get(tau, rom_modes_map.get(str(tau)))
        r_lbl = str(r_tot)
        comp_src = None
        if "rom_r_components" in results:
            comp_map = results["rom_r_components"]
            comp_src = comp_map.get(tau, comp_map.get(str(tau), None))
        if comp_src is not None:
            r_lbl = format_r_triplet((comp_src.get("r_p", np.nan), comp_src.get("r_u", np.nan), comp_src.get("r_s", np.nan)))
        else:
            # Backward-compatible reconstruction when old JSON lacks stored components.
            if "svals_p" in results:
                r_p = choose_rank_from_svals(np.asarray(results["svals_p"], dtype=float), float(tau))
                r_u_stab = float(r_tot) - float(r_p)
                r_lbl = format_r_triplet(r_triplet_from_stabilized_dims(r_u_stab, float(r_p)))

        h1, = ax.plot(
            P_values,
            errs["u_l2_mean"],
            marker="x",
            linestyle="-",
            label="_nolegend_",
            alpha=0.55,
            zorder=2,
        )
        ax.plot(
            P_values,
            errs["u_l2_max"],
            marker="x",
            linestyle="--",
            color=h1.get_color(),
            label="_nolegend_",
            alpha=0.55,
            zorder=2,
        )

        h2, = ax.plot(
            P_values,
            errs["u_hdiv_mean"],
            marker="s",
            linestyle="-",
            color=h1.get_color(),
            label="_nolegend_",
            alpha=0.95,
            markerfacecolor="none",
            markeredgewidth=1.4,
            zorder=3,
        )
        ax.plot(
            P_values,
            errs["u_hdiv_max"],
            marker="s",
            linestyle="--",
            color=h1.get_color(),
            label="_nolegend_",
            alpha=0.95,
            markerfacecolor="none",
            markeredgewidth=1.4,
            zorder=3,
        )

        tau_handles.append(Line2D([0], [0], color=h1.get_color(), linestyle="-", linewidth=1.8, label=rf"$\tau={_tau_to_latex(tau)}$ $(r={r_lbl})$"))

    ax.set_yscale("log")
    ax.set_xlabel(f"Polynomial degree $P$")
    ax.set_ylabel(r"Relative error")
    # ax.set_title("ROM velocity errors vs polynomial degree")
    ax.grid(True, which="both", ls="--", alpha=0.5)

    leg1 = ax.legend(handles=tau_handles, loc="upper right", title="", handlelength=1.6, borderpad=0.4, labelspacing=0.3)
    ax.add_artist(leg1)
    ax.legend(handles=meta_handles, loc="lower left", title="")

    fig.tight_layout()
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_rom_pressure_errors(results, filename="rom_pressure_errors.pdf"):
    P_values = results["P_values"]
    tau_values = results["tau_values"]

    fig, ax = plt.subplots(figsize=(8, 5))

    quantity_handles = []
    style_handles = [
        Line2D([0], [0], color="black", marker="o", linestyle="-", linewidth=1.8, markerfacecolor="black", markeredgecolor="black", label="Mean"),
        Line2D([0], [0], color="black", marker="o", linestyle="--", linewidth=1.8, markerfacecolor="none", markeredgecolor="black", markeredgewidth=1.4, label="Max"),
    ]

    for tau in tau_values:
        rom_errs_map = results["rom_errors"]
        rom_modes_map = results["rom_modes"]
        errs = rom_errs_map.get(tau, rom_errs_map.get(str(tau)))
        r_tot = rom_modes_map.get(tau, rom_modes_map.get(str(tau)))
        r_lbl = str(r_tot)
        comp_src = None
        if "rom_r_components" in results:
            comp_map = results["rom_r_components"]
            comp_src = comp_map.get(tau, comp_map.get(str(tau), None))
        if comp_src is not None:
            r_lbl = format_r_triplet((comp_src.get("r_p", np.nan), comp_src.get("r_u", np.nan), comp_src.get("r_s", np.nan)))
        else:
            # Backward-compatible reconstruction when old JSON lacks stored components.
            if "svals_p" in results:
                r_p = choose_rank_from_svals(np.asarray(results["svals_p"], dtype=float), float(tau))
                r_u_stab = float(r_tot) - float(r_p)
                r_lbl = format_r_triplet(r_triplet_from_stabilized_dims(r_u_stab, float(r_p)))

        h, = ax.plot(
            P_values,
            errs["p_l2_mean"],
            marker="o",
            linestyle="-",
            label=rf"$\tau={_tau_to_latex(tau)}$ $(r={r_lbl})$",
        )
        ax.plot(
            P_values,
            errs["p_l2_max"],
            marker="o",
            linestyle="--",
            color=h.get_color(),
            label="_nolegend_",
            markerfacecolor="none",
            markeredgewidth=1.4,
        )
        quantity_handles.append(Line2D([0], [0], color=h.get_color(), linestyle="-", linewidth=1.8, label=rf"$\tau={_tau_to_latex(tau)}$ $(r={r_lbl})$"))

    ax.set_yscale("log")
    ax.set_xlabel(f"Polynomial degree $P$")
    ax.set_ylabel(r"Relative $L^2$ error")
    # ax.set_title("ROM pressure errors vs polynomial degree")
    ax.grid(True, which="both", ls="--", alpha=0.5)

    leg1 = ax.legend(handles=quantity_handles, loc="upper right", title="")
    ax.add_artist(leg1)
    ax.legend(handles=style_handles, loc="lower left", title="")

    fig.tight_layout()
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_l2_fit_field_and_operator_errors(results, filename="l2_fit_errors.pdf"):
    P_values = np.asarray(results["P_values"], dtype=int)

    fig, ax = plt.subplots(figsize=(8, 5))

    # Plot operator errors first.
    h_op, = ax.plot(
        P_values,
        results["l2fit_operator_int"],
        marker="o",
        linestyle="-",
        # label=r"$\| e_{\mathbf{A}}(\boldsymbol{\mu}) \|_{F}$",
        label=r"$\| e_{\mathbf{A}} \|_{F}$",
    )
    ax.plot(
        P_values,
        results["l2fit_operator_max"],
        marker="o",
        linestyle="--",
        color=h_op.get_color(),
        label="_nolegend_",
    )

    h_p_l2, = ax.plot(
        P_values,
        results["l2fit_p_l2_int"],
        marker="o",
        markersize=4.2,
        linestyle="-",
        zorder=2,
        label=r"$\| e_p \|_{L^2(\Omega)}$",
    )
    ax.plot(
        P_values,
        results["l2fit_p_l2_max"],
        marker="o",
        linestyle="--",
        mfc="none",
        mec=h_p_l2.get_color(),
        mew=1.3,
        color=h_p_l2.get_color(),
        label="_nolegend_",
    )

    h_u_l2, = ax.plot(
        P_values,
        results["l2fit_u_l2_int"],
        marker="x",
        linestyle="-",
        label=r"$\| e_{\mathbf{u}} \|_{L^2(\Omega)}$",
    )
    ax.plot(
        P_values,
        results["l2fit_u_l2_max"],
        marker="x",
        linestyle="--",
        color=h_u_l2.get_color(),
        label="_nolegend_",
    )

    h_u_hdiv, = ax.plot(
        P_values,
        results["l2fit_u_hdiv_int"],
        marker="s",
        linestyle="-",
        markerfacecolor="none",
        markeredgewidth=1.3,
        label=r"$\| e_{\mathbf{u}} \|_{H(\mathrm{div})}$",
    )
    ax.plot(
        P_values,
        results["l2fit_u_hdiv_max"],
        marker="s",
        linestyle="--",
        markerfacecolor="none",
        markeredgewidth=1.3,
        color=h_u_hdiv.get_color(),
        label="_nolegend_",
    )

    all_series = [
        np.asarray(results["l2fit_operator_int"], dtype=float),
        np.asarray(results["l2fit_operator_max"], dtype=float),
        np.asarray(results["l2fit_u_l2_int"], dtype=float),
        np.asarray(results["l2fit_u_l2_max"], dtype=float),
        np.asarray(results["l2fit_u_hdiv_int"], dtype=float),
        np.asarray(results["l2fit_u_hdiv_max"], dtype=float),
        np.asarray(results["l2fit_p_l2_int"], dtype=float),
        np.asarray(results["l2fit_p_l2_max"], dtype=float),
    ]
    y_max = max(float(np.max(arr)) for arr in all_series)
    y_min = min(float(np.min(arr[arr > 0])) for arr in all_series if np.any(arr > 0))

    ax.set_yscale("log")
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(P_values)
    ax.set_xlabel("Polynomial degree $P$")
    ax.set_ylabel("Relative error")
    ax.grid(True, which="both", ls="--", alpha=0.5)

    quantity_handles = [h_op, h_p_l2, h_u_l2, h_u_hdiv]
    style_handles = [
        Line2D([0], [0], color="black", linestyle="-", linewidth=1.8, label="Integrated"),
        Line2D([0], [0], color="black", linestyle="--", linewidth=1.8, label="Max"),
    ]
    leg1 = ax.legend(handles=quantity_handles, loc="upper right", title="")
    ax.add_artist(leg1)
    ax.legend(handles=style_handles, loc="lower left", title="")

    fig.tight_layout()
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_discarded_energy(results, filename="discarded_energy.pdf"):
    svals_u = results["svals_u"]
    svals_p = results["svals_p"]

    ru, disc_u = discarded_energy_curve(svals_u)
    rp, disc_p = discarded_energy_curve(svals_p)

    y_floor = 1e-8

    def last_visible_index(r, values, floor):
        idx = np.where(np.asarray(values, dtype=float) >= floor)[0]
        if len(idx) == 0:
            return int(r[0])
        return int(r[idx[-1]])

    x_max = max(
        last_visible_index(ru, disc_u, y_floor),
        last_visible_index(rp, disc_p, y_floor),
        1,
    )

    fig, ax_left = plt.subplots(figsize=(7, 5))

    h_vel, = ax_left.plot(ru, disc_u, linestyle="-", label="Velocity")
    h_pres, = ax_left.plot(rp, disc_p, linestyle="-", label="Pressure")

    ax_left.set_xlabel("Number of modes $(r_p,r_{\mathbf{u}})$")
    ax_left.set_ylabel("Relative discarded energy")
    # ax_left.set_title("Relative discarded energy")
    ax_left.grid(True, which="both", ls="--", alpha=0.5)
    ax_left.legend()

    all_series = [disc_u, disc_p]
    y_max = max(float(np.max(arr)) for arr in all_series)

    ax_left.set_yscale("log")
    ax_left.set_ylim(y_floor, y_max)
    ax_left.set_xlim(0, x_max)

    fig.tight_layout()
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)



def _mesh_vertex_triangulation(mesh, deformation_gf=None):
    vertices = sorted(list(mesh.vertices), key=lambda v: v.nr)

    vx_ref = np.array([float(v.point[0]) for v in vertices], dtype=float)
    vy_ref = np.array([float(v.point[1]) for v in vertices], dtype=float)

    if deformation_gf is None:
        vx = vx_ref.copy()
        vy = vy_ref.copy()
    else:
        vx = np.empty_like(vx_ref)
        vy = np.empty_like(vy_ref)
        for i, (xv, yv) in enumerate(zip(vx_ref, vy_ref)):
            try:
                dxy = np.asarray(deformation_gf(mesh(xv, yv)), dtype=float).reshape(-1)
            except Exception:
                dxy = np.asarray(deformation_gf(xv, yv), dtype=float).reshape(-1)

            if dxy.size < 2:
                raise RuntimeError("Deformation field evaluation did not return a 2D displacement.")

            vx[i] = xv + float(dxy[0])
            vy[i] = yv + float(dxy[1])

    triangles = []
    for el in mesh.Elements(VOL):
        vids = [int(v.nr) for v in el.vertices]
        if len(vids) == 3:
            triangles.append(vids)
        elif len(vids) == 4:
            triangles.append([vids[0], vids[1], vids[2]])
            triangles.append([vids[0], vids[2], vids[3]])

    if len(triangles) == 0:
        raise RuntimeError("Could not build a 2D triangulation from the mesh.")

    return vx, vy, np.array(triangles, dtype=int), vx_ref, vy_ref


def _evaluate_pressure_at_vertices(mesh, order, p_vec, vx, vy):
    Q = L2(mesh, order=order - 1)
    p_gf = GridFunction(Q)
    p_gf.vec.FV().NumPy()[:] = np.asarray(p_vec, dtype=float)

    values = np.full(len(vx), np.nan, dtype=float)
    for i, (xv, yv) in enumerate(zip(vx, vy)):
        try:
            values[i] = float(p_gf(mesh(xv, yv)))
        except Exception:
            pass
    return values


def _evaluate_velocity_magnitude_at_vertices(mesh, order, u_vec, vx, vy):
    V = HDiv(mesh, order=order)
    u_gf = GridFunction(V)
    u_gf.vec.FV().NumPy()[:] = np.asarray(u_vec, dtype=float)

    values = np.full(len(vx), np.nan, dtype=float)
    for i, (xv, yv) in enumerate(zip(vx, vy)):
        try:
            val = np.asarray(u_gf(mesh(xv, yv)), dtype=float).reshape(-1)
            values[i] = float(np.linalg.norm(val))
        except Exception:
            pass
    return values



def _evaluate_velocity_vector_at_vertices(mesh, order, u_vec, vx, vy):
    V = HDiv(mesh, order=order)
    u_gf = GridFunction(V)
    u_gf.vec.FV().NumPy()[:] = np.asarray(u_vec, dtype=float)

    ux = np.full(len(vx), np.nan, dtype=float)
    uy = np.full(len(vx), np.nan, dtype=float)
    for i, (xv, yv) in enumerate(zip(vx, vy)):
        try:
            val = np.asarray(u_gf(mesh(xv, yv)), dtype=float).reshape(-1)
            if val.size >= 2:
                ux[i] = float(val[0])
                uy[i] = float(val[1])
        except Exception:
            pass
    return ux, uy

def _latex_sci(val, digits=2):
    if not np.isfinite(val):
        return r"\mathrm{nan}"
    if val == 0:
        return "0"

    exp = int(np.floor(np.log10(abs(val))))
    mant = val / (10 ** exp)
    return rf"{mant:.{digits}f} \times 10^{{{exp}}}"



def format_r_value(r):
    return f"{float(r):.2f}".rstrip("0").rstrip(".")

def format_r_triplet(vals):
    if vals is None:
        return "N/A"
    r_p, r_u, r_s = vals
    return f"[{format_r_value(r_p)},{format_r_value(r_u)},{format_r_value(r_s)}]"

def r_triplet_from_stabilized_dims(r_u_stab, r_p):
    r_p = float(r_p)
    r_s = r_p
    r_u = float(r_u_stab) - r_s
    return (r_p, r_u, r_s)

def _tau_to_latex(tau):
    tau = float(tau)
    if tau == 0.0:
        return "0"
    exp = int(np.round(np.log10(abs(tau))))
    # If not an exact power of 10, fall back to generic scientific formatting.
    if not np.isclose(abs(tau), 10.0**exp, rtol=1e-12, atol=0.0):
        return _latex_sci(tau, digits=2)
    sign = "-" if tau < 0 else ""
    return rf"{sign}10^{{{exp}}}"

def _finite_extrema(arrays):
    finite = [arr[np.isfinite(arr)] for arr in arrays if np.any(np.isfinite(arr))]
    if not finite:
        raise RuntimeError("No finite values available for plotting.")
    merged = np.concatenate(finite)
    return float(np.min(merged)), float(np.max(merged))


def _plot_fem_rom_quantity_for_two_taus(
    experiment,
    mu_plot,
    tau_values,
    fom_out,
    rom_solutions,
    quantity,
    filename,
    P,
):
    experiment.geom.apply(tuple(mu_plot))
    try:
        vx, vy, triangles, vx_ref, vy_ref = _mesh_vertex_triangulation(experiment.mesh, deformation_gf=experiment.geom.deform_gf)
        triang = mtri.Triangulation(vx, vy, triangles=triangles)

        rows = []
        for tau in tau_values:
            rom_entry = rom_solutions[float(tau)]
            rom_sol = rom_entry["solution"]
            r_tot = int(rom_entry["r_tot"])
            row = {
                "tau": float(tau),
                "r_tot": r_tot,
                "r_triplet": rom_entry.get("r_triplet", None),
                "r_label": format_r_triplet(rom_entry.get("r_triplet", None)) if rom_entry.get("r_triplet", None) is not None else str(r_tot),
            }

            if quantity == "pressure":
                fem_vals = _evaluate_pressure_at_vertices(experiment.mesh, experiment.order, fom_out["p"], vx_ref, vy_ref)
                rom_vals = _evaluate_pressure_at_vertices(experiment.mesh, experiment.order, rom_sol["p"], vx_ref, vy_ref)
                quantity_title = "Pressure"

                diff_vals = rom_vals - fem_vals
                valid = np.isfinite(fem_vals) & np.isfinite(rom_vals)
                if np.any(valid):
                    eps = np.linalg.norm(diff_vals[valid]) / max(np.linalg.norm(fem_vals[valid]), 1e-14)
                else:
                    eps = np.nan

                row.update({
                    "fem": fem_vals,
                    "rom": rom_vals,
                    "diff": diff_vals,
                    "eps": float(eps),
                })

            elif quantity == "velocity":
                fem_vals = _evaluate_velocity_magnitude_at_vertices(experiment.mesh, experiment.order, fom_out["u"], vx_ref, vy_ref)
                rom_vals = _evaluate_velocity_magnitude_at_vertices(experiment.mesh, experiment.order, rom_sol["u"], vx_ref, vy_ref)
                fem_ux, fem_uy = _evaluate_velocity_vector_at_vertices(experiment.mesh, experiment.order, fom_out["u"], vx_ref, vy_ref)
                rom_ux, rom_uy = _evaluate_velocity_vector_at_vertices(experiment.mesh, experiment.order, rom_sol["u"], vx_ref, vy_ref)

                err_ux = rom_ux - fem_ux
                err_uy = rom_uy - fem_uy
                err_norm = np.sqrt(err_ux**2 + err_uy**2)

                valid_vec = np.isfinite(fem_ux) & np.isfinite(fem_uy) & np.isfinite(rom_ux) & np.isfinite(rom_uy)
                if np.any(valid_vec):
                    num = np.sqrt(np.sum(err_ux[valid_vec]**2 + err_uy[valid_vec]**2))
                    den = max(np.sqrt(np.sum(fem_ux[valid_vec]**2 + fem_uy[valid_vec]**2)), 1e-14)
                    eps = num / den
                else:
                    eps = np.nan

                row.update({
                    "fem": fem_vals,
                    "rom": rom_vals,
                    "fem_ux": fem_ux,
                    "fem_uy": fem_uy,
                    "rom_ux": rom_ux,
                    "rom_uy": rom_uy,
                    "err_ux": err_ux,
                    "err_uy": err_uy,
                    "err_norm": err_norm,
                    "eps": float(eps),
                })
                quantity_title = "Velocity magnitude"
            else:
                raise ValueError(f"Unknown quantity: {quantity}")

            row["title"] = quantity_title
            rows.append(row)

        field_min, field_max = _finite_extrema([r["fem"] for r in rows] + [r["rom"] for r in rows])

        if quantity == "velocity":
            _, diff_max = _finite_extrema([r["err_norm"] for r in rows])
            diff_min = 0.0
            if diff_max <= 0.0:
                diff_max = 1e-14

            vel_max = 0.0
            for row in rows:
                vel_mag = np.sqrt(row["rom_ux"]**2 + row["rom_uy"]**2)
                fem_mag = np.sqrt(row["fem_ux"]**2 + row["fem_uy"]**2)
                if np.any(np.isfinite(vel_mag)):
                    vel_max = max(vel_max, float(np.nanmax(vel_mag)))
                if np.any(np.isfinite(fem_mag)):
                    vel_max = max(vel_max, float(np.nanmax(fem_mag)))

            span = max(float(np.ptp(vx)), float(np.ptp(vy)), 1e-14)
            target_arrow = 0.18 * span
            vel_scale = max(vel_max / target_arrow, 1e-14)
            diff_scale = max(diff_max / target_arrow, 1e-14)
        else:
            diff_min, diff_max = _finite_extrema([r["diff"] for r in rows])
            diff_lim = max(abs(diff_min), abs(diff_max))
            if diff_lim <= 0.0:
                diff_lim = 1e-14

        fig = plt.figure(figsize=(7.6, 9.8))
        gs = fig.add_gridspec(
            3,
            2,
            left=0.08,
            right=0.86,
            bottom=0.06,
            top=0.975,
            hspace=0.14,
            wspace=0.16,
        )
        title_fs = 11

        def _set_bottom_title(ax, text):
            ax.set_title("")
            ax.text(0.5, -0.06, text, transform=ax.transAxes, ha="center", va="top", fontsize=title_fs)

        ax_rom_1 = fig.add_subplot(gs[1, 0])
        ax_rom_2 = fig.add_subplot(gs[1, 1])
        ax_fem = fig.add_subplot(gs[0, 0])
        ax_diff_1 = fig.add_subplot(gs[2, 0])
        ax_diff_2 = fig.add_subplot(gs[2, 1])

        if quantity == "velocity":
            m_field = None
            m_rom = None
        else:
            m_field = ax_fem.tripcolor(
                triang,
                rows[0]["fem"],
                shading="gouraud",
                cmap="viridis",
                vmin=field_min,
                vmax=field_max,
            )
            m_rom = ax_rom_1.tripcolor(
                triang,
                rows[0]["rom"],
                shading="gouraud",
                cmap="viridis",
                vmin=field_min,
                vmax=field_max,
            )
            ax_rom_2.tripcolor(
                triang,
                rows[1]["rom"],
                shading="gouraud",
                cmap="viridis",
                vmin=field_min,
                vmax=field_max,
            )

        _set_bottom_title(ax_fem, f"FOM solution, {int(fom_out['Nu'] + fom_out['Np'])} DoFs")
        _set_bottom_title(ax_rom_1, rf"ROM solution, r={rows[0]['r_label']}, P={int(P)}")
        _set_bottom_title(ax_rom_2, rf"ROM solution, r={rows[1]['r_label']}, P={int(P)}")

        if quantity == "velocity":
            step = max(1, len(vx) // 120)
            ids = np.arange(0, len(vx), step)

            def _vel_quiver(ax, ux, uy, mag):
                valid = np.isfinite(ux) & np.isfinite(uy) & np.isfinite(mag)
                ids_v = ids[valid[ids]]
                if len(ids_v) == 0:
                    return None
                q = ax.quiver(
                    vx[ids_v],
                    vy[ids_v],
                    ux[ids_v],
                    uy[ids_v],
                    mag[ids_v],
                    cmap="viridis",
                    angles="xy",
                    scale_units="xy",
                    scale=vel_scale,
                    width=0.0025,
                    headwidth=3.2,
                    headlength=4.2,
                )
                q.set_clim(field_min, field_max)
                return q

            m_field = _vel_quiver(
                ax_fem,
                rows[0]["fem_ux"],
                rows[0]["fem_uy"],
                np.sqrt(rows[0]["fem_ux"]**2 + rows[0]["fem_uy"]**2),
            )
            m_rom = _vel_quiver(
                ax_rom_1,
                rows[0]["rom_ux"],
                rows[0]["rom_uy"],
                np.sqrt(rows[0]["rom_ux"]**2 + rows[0]["rom_uy"]**2),
            )
            _vel_quiver(
                ax_rom_2,
                rows[1]["rom_ux"],
                rows[1]["rom_uy"],
                np.sqrt(rows[1]["rom_ux"]**2 + rows[1]["rom_uy"]**2),
            )

            # Difference panels: vector error arrows. Length from error vector, color from error norm.
            def _error_quiver(ax, row):
                valid = np.isfinite(row["err_ux"]) & np.isfinite(row["err_uy"]) & np.isfinite(row["err_norm"])
                ids_v = ids[valid[ids]]
                if len(ids_v) == 0:
                    return None
                q = ax.quiver(
                    vx[ids_v],
                    vy[ids_v],
                    row["err_ux"][ids_v],
                    row["err_uy"][ids_v],
                    row["err_norm"][ids_v],
                    cmap="RdBu",
                    angles="xy",
                    scale_units="xy",
                    scale=diff_scale,
                    width=0.0030,
                    headwidth=3.5,
                    headlength=4.5,
                    minlength=0.0,
                )
                q.set_clim(diff_min, diff_max)
                return q

            m_diff = _error_quiver(ax_diff_1, rows[0])
            _error_quiver(ax_diff_2, rows[1])
            _set_bottom_title(ax_diff_1, rf"ROM-FOM difference, r={rows[0]['r_label']}, P={int(P)}")
            _set_bottom_title(ax_diff_2, rf"ROM-FOM difference, r={rows[1]['r_label']}, P={int(P)}")
        else:
            m_diff = ax_diff_1.tripcolor(
                triang,
                rows[0]["diff"],
                shading="gouraud",
                cmap="RdBu",
                vmin=-diff_lim,
                vmax=diff_lim,
            )
            ax_diff_2.tripcolor(
                triang,
                rows[1]["diff"],
                shading="gouraud",
                cmap="RdBu",
                vmin=-diff_lim,
                vmax=diff_lim,
            )
            _set_bottom_title(ax_diff_1, rf"ROM-FOM difference, r={rows[0]['r_label']}, P={int(P)}")
            _set_bottom_title(ax_diff_2, rf"ROM-FOM difference, r={rows[1]['r_label']}, P={int(P)}")

        if quantity == "pressure":
            # Use one shared level-set definition for all pressure panels (FOM + ROMs).
            # Difference panels are intentionally excluded from contour overlays.
            contour_targets = [
                (ax_fem, rows[0]["fem"]),
                (ax_rom_1, rows[0]["rom"]),
                (ax_rom_2, rows[1]["rom"]),
            ]
            n_contours = 12
            if abs(field_max - field_min) > 1e-14:
                levels = np.linspace(field_min, field_max, n_contours)
                for ax_c, data_c in contour_targets:
                    finite = np.asarray(data_c, dtype=float)
                    finite = finite[np.isfinite(finite)]
                    if len(finite) < 2:
                        continue
                    ax_c.tricontour(
                        triang,
                        data_c,
                        levels=levels,
                        colors="black",
                        linewidths=0.35,
                        alpha=0.45,
                    )

        for ax in [ax_fem, ax_rom_1, ax_rom_2, ax_diff_1, ax_diff_2]:
            ax.set_aspect("equal")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

        p1 = ax_rom_1.get_position()
        p2 = ax_rom_2.get_position()
        p0 = ax_fem.get_position()
        x_center = 0.5 * (p1.x0 + p2.x1)
        ax_fem.set_position([x_center - 0.5 * p1.width, p0.y0, p1.width, p0.height])

        def _add_side_cbar(mappable, ax_anchor, label=None):
            if mappable is None:
                return
            pos = ax_anchor.get_position()
            cax = fig.add_axes([
                pos.x1 + 0.012,
                pos.y0 + 0.03 * pos.height,
                0.015,
                0.94 * pos.height,
            ])
            cb = fig.colorbar(mappable, cax=cax)
            if label is not None:
                cb.set_label(label)

        # _add_side_cbar(m_field, ax_fem, f"FOM {rows[0]['title']}")
        # _add_side_cbar(m_rom, ax_rom_2, f"ROM {rows[0]['title']}")
        _add_side_cbar(m_field, ax_fem, None)
        _add_side_cbar(m_rom, ax_rom_2, None)
        # Keep difference colorbar label disabled for now.
        _add_side_cbar(m_diff, ax_diff_2, None)

        fig.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)
    finally:
        experiment.geom.clear()
def plot_fem_rom_solution_comparison_for_taus(
    experiment,
    mu_train,
    W_train,
    mu_plot,
    tau_values,
    P=9,
    outdir=".",
):
    if len(tau_values) != 2:
        raise ValueError("Please provide exactly two tau values.")

    if outdir:
        os.makedirs(outdir, exist_ok=True)

    # Memory-efficient path: snapshots first, then streaming reduced-operator assembly per tau.
    Uu_train, Up_train, Nu, Np = collect_snapshot_data(experiment, mu_train)

    fom_out = experiment.solve_full(tuple(mu_plot), collect_system=False)

    rom_solutions = {}
    for tau in tau_values:
        rom = fit_rom_streaming(
            experiment=experiment,
            mu_train=mu_train,
            W_train=W_train,
            Uu_train=Uu_train,
            Up_train=Up_train,
            Nu=Nu,
            Np=Np,
            P=int(P),
            tau=float(tau),
        )
        rom_sol = rom.solve_online(tuple(mu_plot))
        rom_solutions[float(tau)] = {
            "solution": rom_sol,
            "r_tot": int(rom.r_tot),
            "r_triplet": r_triplet_from_stabilized_dims(rom.r_u_stab, rom.r_p),
        }

    _plot_fem_rom_quantity_for_two_taus(
        experiment=experiment,
        mu_plot=tuple(mu_plot),
        tau_values=[float(tau_values[0]), float(tau_values[1])],
        fom_out=fom_out,
        rom_solutions=rom_solutions,
        quantity="pressure",
        filename=os.path.join(outdir, f"fem_rom_pressure_two_taus_P{int(P)}.pdf"),
        P=int(P),
    )

    _plot_fem_rom_quantity_for_two_taus(
        experiment=experiment,
        mu_plot=tuple(mu_plot),
        tau_values=[float(tau_values[0]), float(tau_values[1])],
        fom_out=fom_out,
        rom_solutions=rom_solutions,
        quantity="velocity",
        filename=os.path.join(outdir, f"fem_rom_velocity_two_taus_P{int(P)}.pdf"),
        P=int(P),
    )

def plot_speedup_vs_polynomial_degree(speed_results, filename="rom_speedup_vs_polynomial_degree.pdf"):
    P_values = np.asarray(speed_results["P_values"], dtype=int)
    tau_values = np.asarray(speed_results["tau_values"], dtype=float)

    fig, ax = plt.subplots(figsize=(8, 5))

    for tau in tau_values:
        tau_key = float(tau)
        r_tot = int(speed_results["rom_modes_by_tau"][tau_key][0])
        lbl = rf"$r={r_tot}$"
        ax.plot(
            P_values,
            speed_results["speedup_by_tau"][tau_key],
            marker="o",
            label=lbl,
        )


    ax.set_xticks(P_values)
    ax.set_xlabel("Polynomial degree $P$")
    ax.set_ylabel("Speed-up (FOM / ROM)")
    ax.grid(True, which="both", ls="--", alpha=0.5)
    ax.legend()

    fig.tight_layout()
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)










<>:1668: SyntaxWarning: invalid escape sequence '\m'
<>:1668: SyntaxWarning: invalid escape sequence '\m'
C:\Users\henni\AppData\Local\Temp\ipykernel_26492\2815182777.py:1668: SyntaxWarning: invalid escape sequence '\m'
  ax_left.set_xlabel("Number of modes $(r_p,r_{\mathbf{u}})$")


In [2]:
if __name__ == "__main__":
    mesh = Mesh(unit_square.GenerateMesh(maxh=0.16))

    domain = ParameterDomain(mu1_range=(-0.3, 0.3), mu2_range=(-0.3, 0.3))
    deformation = TopRightDeformation()
    darcy_model = DarcyModel(K=1.0)

    experiment = DarcyROMExperiment(
        mesh=mesh,
        order=2,
        darcy_model=darcy_model,
        domain=domain,
        deformation_model=deformation,
    )

    # training set
    # mu_train, W_train = sample_parameters(method="grid", N=16, domain=domain)
    mu_train, W_train = sample_parameters(method="grid", N=8, domain=domain)

    run_speed_test = True
    run_convergence = True

    P_values = list(range(10))  # 0 to 9 inclusive
    P_values = [4,5]  # 0 to 9 inclusive
    tau_values = [1e-1,1e-2,1e-3,1e-6]
    tau_values = [1e-1,1e-2]
    n_test = 16
    # n_test = 256
    random_seed = 0
    speed_repeats = 10

    speed_results = None
    results = None

    if run_convergence:
        results = run_convergence_study(
            experiment=experiment,
            domain=domain,
            P_values=P_values,
            tau_values=tau_values,
            mu_train=mu_train,
            W_train=W_train,
            n_test_random=n_test,
            random_seed=random_seed,
            gl_n1d=16,
            measure_speed=run_speed_test,
            speed_repeats=speed_repeats,
        )

        if run_speed_test:
            speed_results = speed_results_from_convergence(results)

    elif run_speed_test:
        speed_results = benchmark_rom_speedup_vs_polynomial_degree(
            experiment=experiment,
            domain=domain,
            mu_train=mu_train,
            W_train=W_train,
            P_values=P_values,
            tau_values=tau_values,
            n_test=n_test,
            random_seed=random_seed,
            n_repeats=speed_repeats,
        )

    outdir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    results_json_path = os.path.join(outdir, "rom_deformedv3_results.json")
    save_results_bundle_json(
        results_json_path,
        results=results,
        speed_results=speed_results,
        extra={
            "P_values": P_values,
            "tau_values": tau_values,
            "run_speed_test": run_speed_test,
            "run_convergence": run_convergence,
        },
    )
    print("Saved results JSON:", results_json_path)







H(div) diagnostic: ||Mdiv||_F / ||Mu||_F = 4.800e+02


Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

Saved results JSON: c:\Users\henni\OneDrive\Dokumenter\GitHub\Masters-2026\examples\deformed_square_stationary\rom_deformedv3_results.json


In [3]:
def _restore_numeric_keys(obj):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            nk = k
            if isinstance(k, str):
                try:
                    nk = float(k)
                except ValueError:
                    nk = k
            out[nk] = _restore_numeric_keys(v)
        return out
    return obj

json_path = "rom_deformedv3_results.json"
bundle = load_results_bundle_json(json_path)
results = _restore_numeric_keys(bundle.get("results", None))
speed_results = _restore_numeric_keys(bundle.get("speed_results", None))
outdir = "plots"
# outdir = ""
if outdir:
    os.makedirs(outdir, exist_ok=True)

def outpath(name):
    return os.path.join(outdir, name) if outdir else name

plot_rom_velocity_errors(results, outpath("rom_velocity_errors.pdf"))
plot_rom_pressure_errors(results, outpath("rom_pressure_errors.pdf"))
plot_l2_fit_field_and_operator_errors(results, outpath("l2_fit_errors.pdf"))
plot_discarded_energy(results, outpath("discarded_energy.pdf"))
plot_speedup_vs_polynomial_degree(speed_results, outpath("rom_speedup_vs_polynomial_degree.pdf"))


# FEM vs ROM field plots for two taus, with P
# tau_plot_values = [float(tau_values[0]), float(tau_values[-1])]
# tau_plot_values = [1e-1,1e-2]
# mu_plot = (0.3, 0.3)
# plot_fem_rom_solution_comparison_for_taus(
#     experiment=experiment,
#     mu_train=mu_train,
#     W_train=W_train,
#     mu_plot=mu_plot,
#     tau_values=tau_plot_values,
#     P=3,
#     outdir=outdir,
# )
